# Email Spam Classifier Using Machine Learning

## CODTECH Internship Task 2

This notebook demonstrates a complete NLP machine learning pipeline for classifying emails/messages as Spam or Ham.

In [ ]:
import pandas as pd
import numpy as np
import string
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)
sns.set_theme(style="whitegrid")

### 1. Load the Dataset

In [ ]:
data = pd.read_csv('data/spam.csv')
display(data.head())

### 2. Text Preprocessing

In [ ]:
def preprocess_text(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    stop_words = set(stopwords.words('english'))
    tokens = text.split()
    cleaned_tokens = [word for word in tokens if word not in stop_words]
    return " ".join(cleaned_tokens)

data = data.dropna()
data['cleaned_text'] = data['text'].apply(preprocess_text)
display(data[['text', 'cleaned_text']].head())

### 3. Feature Extraction (TF-IDF)

In [ ]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(data['cleaned_text'])
y = data['label']

### 4. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training Set: {X_train.shape}")
print(f"Testing Set: {X_test.shape}")

### 5. Model Training (Naive Bayes)

In [ ]:
model = MultinomialNB()
model.fit(X_train, y_train)

### 6. Model Evaluation

In [ ]:
y_pred = model.predict(X_test)

print(f"Accuracy Score: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

### 7. Confusion Matrix Visualization

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=model.classes_, yticklabels=model.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

### 8. Test the Prediction Function

In [ ]:
def predict_spam(text):
    processed = preprocess_text(text)
    vec = vectorizer.transform([processed])
    return model.predict(vec)[0]

sample_1 = "Congratulations! You've won a free vacation. Click here to claim."
sample_2 = "Hey, let's catch up for lunch tomorrow."

print(f"'{sample_1}' -> {predict_spam(sample_1).upper()}")
print(f"'{sample_2}' -> {predict_spam(sample_2).upper()}")